# NexusGTM store explorer

Reads `nexusgtm.db` and shows what the orchestrator writes: one orchestration,
its agent runs, the planner's decisions, and every cost event.

Read-only — nothing here writes. Stdlib only, so it needs no packages beyond the kernel.

The four tables:

| table | one row is | why it exists |
|---|---|---|
| `orchestrations` | one run | the final blackboard, outcome, total cost |
| `runs` | one agent invocation | input, output, status per step |
| `decisions` | one planner step | *why* this agent, out of which candidates |
| `cost_events` | one charge | agent spend **and** planner spend |

In [1]:
import json
import sqlite3
from pathlib import Path


def find_db(name="nexusgtm.db"):
    """Walk up from the notebook (or cwd) until the db turns up."""
    start = Path.cwd().resolve()
    for base in (start, *start.parents):
        hit = base / name
        if hit.exists():
            return hit
    raise FileNotFoundError(f"{name} not found from {start}")


DB = find_db()
conn = sqlite3.connect(DB)
conn.row_factory = sqlite3.Row


def q(sql, params=()):
    return conn.execute(sql, params).fetchall()


def show(row, width=400, title=None):
    """Print one sqlite3.Row as aligned key: value lines, long values clipped."""
    if row is None:
        print("  (no row)")
        return
    if title:
        print(title)
    pad = max(len(k) for k in row.keys())
    for k in row.keys():
        v = str(row[k])
        if len(v) > width:
            v = v[:width] + f"  ... [+{len(str(row[k])) - width} chars]"
        print(f"  {k:<{pad}}  {v}")


def blob(value, limit=None):
    """Pretty-print a JSON text column; fall back to the raw string."""
    try:
        text = json.dumps(json.loads(value), indent=2)
    except (TypeError, ValueError):
        text = str(value)
    print(text if limit is None else text[:limit])


print(f"db: {DB}")

db: D:\Atreya\College\Projects\NexusGTM\nexusgtm.db


## Tables and row counts

`sqlite_sequence` is SQLite's own bookkeeping for `AUTOINCREMENT` columns, not ours.

In [2]:
TABLES = [
    r["name"]
    for r in q("select name from sqlite_master where type='table' order by name")
]

for t in TABLES:
    n = q(f'select count(*) as n from "{t}"')[0]["n"]
    print(f"{t:<16} {n:>6} rows")

cost_events         634 rows
decisions           366 rows
orchestrations       89 rows
runs                289 rows
sqlite_sequence       2 rows


## Schema

Note which columns are nullable — `cost_events.run_id` is, deliberately: a planner
charge has no run behind it.

In [3]:
for t in TABLES:
    if t.startswith("sqlite_"):
        continue
    print(f"--- {t} ---")
    for c in q(f'pragma table_info("{t}")'):
        flags = []
        if c["pk"]:
            flags.append("pk")
        flags.append("not null" if c["notnull"] else "nullable")
        print(f"  {c['name']:<20} {c['type'] or '?':<10} {', '.join(flags)}")
    print()

--- cost_events ---
  id                   INTEGER    pk, nullable
  orchestration_id     TEXT       not null
  run_id               TEXT       nullable
  department           TEXT       not null
  agent                TEXT       not null
  amount_usd           REAL       not null
  timestamp            TEXT       not null

--- decisions ---
  id                   INTEGER    pk, nullable
  orchestration_id     TEXT       not null
  step_no              INTEGER    not null
  chosen_agent         TEXT       nullable
  rationale            TEXT       not null
  candidates_considered TEXT       nullable
  timestamp            TEXT       not null
  flow                 TEXT       nullable

--- orchestrations ---
  id                   TEXT       pk, nullable
  crm_reference_id     TEXT       not null
  status               TEXT       not null
  outcome              TEXT       nullable
  started_at           TEXT       not null
  finished_at          TEXT       nullable
  final_result       

## Pick one orchestration

Most recent completed run that carries a flow label. Everything below is scoped to it,
so the four records line up instead of being four unrelated rows.

In [4]:
picked = q(
    "select * from orchestrations "
    "where flow is not null and status = 'completed' "
    "order by started_at desc limit 1"
)

if not picked:  # nothing seeded with a flow yet -- take whatever exists
    picked = q("select * from orchestrations order by started_at desc limit 1")

orch = picked[0]
OID = orch["id"]
print(f"orchestration {OID}\n")
show(orch)

orchestration 15ad9c0c-6403-447d-a835-fbc744c90d57

  id                15ad9c0c-6403-447d-a835-fbc744c90d57
  crm_reference_id  Northwind Logistics
  status            completed
  outcome           qualified
  started_at        2026-08-09T22:47:35.456368+00:00
  finished_at       2026-08-09T22:48:13.893324+00:00
  final_result      {"lead": {"name": "Priya Raman", "email": "priya.raman@northwindlogistics.com", "company": "Northwind Logistics", "source": "console"}, "crm_account": {"exists": true, "record_id": "532354038489", "object_type": "contacts", "lifecycle_stage": "lead", "owner": "", "matched_on": "contacts.email"}, "firmographics": {"company_name": "Northwind Logistics", "domain": "northwindlogistics.com", "industry"  ... [+4625 chars]
  total_cost_usd    0.07878125
  flow              inbound_lead_qualification


### `orchestrations.final_result` — the blackboard at the end

Every key here was merged in by an agent that returned `status: "ok"`, except the
seed keys the run started with.

In [5]:
blob(orch["final_result"])

{
  "lead": {
    "name": "Priya Raman",
    "email": "priya.raman@northwindlogistics.com",
    "company": "Northwind Logistics",
    "source": "console"
  },
  "crm_account": {
    "exists": true,
    "record_id": "532354038489",
    "object_type": "contacts",
    "lifecycle_stage": "lead",
    "owner": "",
    "matched_on": "contacts.email"
  },
  "firmographics": {
    "company_name": "Northwind Logistics",
    "domain": "northwindlogistics.com",
    "industry": "Transportation, Logistics, Supply Chain and Storage",
    "employee_count": 75,
    "revenue_band": "$10M-$50M",
    "hq_region": "United States",
    "free_email_domain": false,
    "confidence": 0.34,
    "source": "inferred"
  },
  "fit_score": 72,
  "reasons": [
    "Company appears to be a B2B logistics/transportation services provider, which fits the broader \u2018software and services\u2019 ICP category, though it is more services-heavy than software-native.",
    "Employee count of 75 (firmographics.source = 'inferr

## One record from each table

Step 1 of the run above, seen from three angles: what the planner decided, what it
cost, what the agent did.

In [6]:
one = {
    "orchestrations": q("select * from orchestrations where id = ?", (OID,)),
    "decisions": q(
        "select * from decisions where orchestration_id = ? order by step_no limit 1",
        (OID,),
    ),
    "runs": q(
        "select * from runs where orchestration_id = ? order by step_no limit 1",
        (OID,),
    ),
    "cost_events": q(
        "select * from cost_events where orchestration_id = ? order by id limit 1",
        (OID,),
    ),
}

for name, rows in one.items():
    print(f"=== {name} ===")
    show(rows[0] if rows else None)
    print()

=== orchestrations ===
  id                15ad9c0c-6403-447d-a835-fbc744c90d57
  crm_reference_id  Northwind Logistics
  status            completed
  outcome           qualified
  started_at        2026-08-09T22:47:35.456368+00:00
  finished_at       2026-08-09T22:48:13.893324+00:00
  final_result      {"lead": {"name": "Priya Raman", "email": "priya.raman@northwindlogistics.com", "company": "Northwind Logistics", "source": "console"}, "crm_account": {"exists": true, "record_id": "532354038489", "object_type": "contacts", "lifecycle_stage": "lead", "owner": "", "matched_on": "contacts.email"}, "firmographics": {"company_name": "Northwind Logistics", "domain": "northwindlogistics.com", "industry"  ... [+4625 chars]
  total_cost_usd    0.07878125
  flow              inbound_lead_qualification

=== decisions ===
  id                     358
  orchestration_id       15ad9c0c-6403-447d-a835-fbc744c90d57
  step_no                1
  chosen_agent           crm_lookup
  rationale            

## The decision trail

The path is chosen at runtime, so this is the only record of why the run went where
it did. `candidates_considered` is the action space the planner was choosing from —
every registered agent, not a shortlist filtered by flow.

In [7]:
for d in q(
    "select * from decisions where orchestration_id = ? order by step_no", (OID,)
):
    print(f"step {d['step_no']}  ->  {d['chosen_agent'] or '(none: finishing)'}")
    print(f"  flow: {d['flow']}")
    print(f"  why:  {d['rationale']}")
    print()

step 1  ->  crm_lookup
  flow: inbound_lead_qualification
  why:  This is an inbound lead that may already exist in our CRM under the same account. Per the crm_context_first pattern, we should cheaply check for an existing account, owner, or open deal before spending on enrichment.

step 2  ->  clay_enrich
  flow: inbound_lead_qualification
  why:  We have a known CRM contact but no firmographics yet. Next, we should enrich the inbound lead via Clay to get real firmographics and buying signals before scoring and routing.

step 3  ->  enrichment
  flow: inbound_lead_qualification
  why:  Clay enrichment failed due to invalid credentials, so we should fall back to the cheaper inference-based enrichment to get firmographics before scoring and routing.

step 4  ->  scoring
  flow: inbound_lead_qualification
  why:  We now have firmographics from enrichment, so the next step in standard_qualification is to score the lead against the ICP before routing or drafting outreach.

step 5  ->  rout

## The runs those decisions produced

A `status` other than `ok` is not a crash — it goes back to the planner as an
observation and the next step re-plans around it.

In [8]:
for r in q("select * from runs where orchestration_id = ? order by step_no", (OID,)):
    print(f"step {r['step_no']}  {r['department']}.{r['agent']}  [{r['status']}]")
    print(f"  in:  {str(r['input'])[:200]}")
    print(f"  out: {str(r['output'])[:200]}")
    print()

step 1  revops.crm_lookup  [ok]
  in:  {"email": "priya.raman@northwindlogistics.com", "domain": "northwindlogistics.com"}
  out: {"crm_account": {"exists": true, "record_id": "532354038489", "object_type": "contacts", "lifecycle_stage": "lead", "owner": "", "matched_on": "contacts.email"}}

step 2  revops.clay_enrich  [error]
  in:  {"lead": {"name": "Priya Raman", "email": "priya.raman@northwindlogistics.com", "company": "Northwind Logistics", "source": "console"}}
  out: {"error": "VendorError: Clay rejected the credentials (401). Check the API key in this process's environment."}

step 3  marketing.enrichment  [ok]
  in:  {"lead": {"name": "Priya Raman", "email": "priya.raman@northwindlogistics.com", "company": "Northwind Logistics", "source": "console"}}
  out: {"firmographics": {"company_name": "Northwind Logistics", "domain": "northwindlogistics.com", "industry": "Transportation, Logistics, Supply Chain and Storage", "employee_count": 75, "revenue_band": "

step 4  revops.scorin

## Cost

Agents hold no db handle — their spend rides back to the orchestrator inside the tool
result. The planner is metered too, as `orchestrator/planner` with a null `run_id`,
which is why the orchestration total sums this table rather than `runs.cost_usd`.

In [9]:
events = q(
    "select * from cost_events where orchestration_id = ? order by id", (OID,)
)

for e in events:
    tag = "planner" if e["run_id"] is None else f"run {e['run_id'][:8]}"
    print(f"  {e['department']:<14} {e['agent']:<18} ${e['amount_usd']:.6f}   {tag}")

summed = sum(e["amount_usd"] for e in events)
runs_only = sum(e["amount_usd"] for e in events if e["run_id"] is not None)

print(f"\n  cost_events total       ${summed:.6f}")
print(f"  orchestrations.total    ${orch['total_cost_usd']:.6f}")
print(f"  agents only (no planner) ${runs_only:.6f}  <- what a runs-only rollup would miss")

  orchestrator   planner            $0.005998   planner
  orchestrator   planner            $0.005914   planner
  orchestrator   planner            $0.006026   planner
  marketing      enrichment         $0.001021   run 3cfd38a5
  orchestrator   planner            $0.006746   planner
  revops         scoring            $0.003172   run ee72c8ab
  orchestrator   planner            $0.006914   planner
  revops         routing            $0.001120   run a7a383db
  orchestrator   planner            $0.010031   planner
  sales          outreach_draft     $0.002887   run 17110586
  orchestrator   planner            $0.009854   planner
  sales          send_outreach      $0.001195   run 87a5ad5f
  orchestrator   planner            $0.010382   planner
  orchestrator   planner            $0.007520   planner

  cost_events total       $0.078781
  orchestrations.total    $0.078781
  agents only (no planner) $0.009396  <- what a runs-only rollup would miss


## Across the whole store

Which agents actually get chosen, and how the runs end up.

In [10]:
print("outcomes:")
for r in q(
    "select status, outcome, count(*) as n from orchestrations "
    "group by status, outcome order by n desc"
):
    print(f"  {r['status']:<16} {str(r['outcome']):<16} {r['n']:>4}")

print("\nagents chosen:")
for r in q(
    "select department, agent, count(*) as n, "
    "sum(status = 'ok') as ok from runs group by department, agent order by n desc"
):
    print(f"  {r['department']:<18} {r['agent']:<20} {r['n']:>4} runs, {r['ok']:>4} ok")

print("\nflows:")
for r in q(
    "select flow, count(*) as n from orchestrations group by flow order by n desc"
):
    print(f"  {str(r['flow']):<32} {r['n']:>4}")

outcomes:
  completed        qualified          46
  completed        disqualified       29
  halted_budget    None                6
  halted_steps     None                6
  halted_loop      None                2

agents chosen:
  revops             scoring                85 runs,   84 ok
  marketing          enrichment             71 runs,   69 ok
  revops             routing                52 runs,   52 ok
  sales              outreach_draft         52 runs,   52 ok
  revops             crm_sync                7 runs,    7 ok
  sales              send_outreach           7 runs,    7 ok
  revops             clay_enrich             5 runs,    0 ok
  revops             crm_lookup              5 runs,    4 ok
  customer_success   churn_detection         2 runs,    1 ok
  marketing          run_ad_campaigns        2 runs,    2 ok
  customer_success   engagement_tracking     1 runs,    1 ok

flows:
  None                               81
  inbound_lead_qualification          3
  churn_sa

In [11]:
conn.close()